# Fit Linear Model via Limma Analysis: REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import combinations

from inmoose import limma
import numpy as np
import pandas as pd
from patsy import dmatrix

from hk1_r12ter_analysis.config import (
    GENE_ID,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.data.transform import transform_log10
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)
from hk1_r12ter_analysis.modeling.linear_model import fit_limma_model_for_genotype

## REDS Recall

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
norm_str = "median-log2-pareto"

### Load data

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()

# Handle group key in index by assigning to metadata and renaming samples
if group_key is not None:
    df_all_data = df_all_data.reset_index(drop=False).set_index(sample_key)
    df_all_data.index = pd.Index(
        [f"{sample_id}-{group_id}" for sample_id, group_id in df_all_data[group_key].items()],
        name="-".join((sample_key, group_key)),
    )
    df_all_data.columns = pd.MultiIndex.from_tuples(
        [("Metadata", group_key) if col == (group_key, "") else col for col in df_all_data.columns]
    )
df_all_data

### Load annotations for mapping SNP IDs to shorthands for file/directory names

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
# Change periods to semicolons when using RSID in file/directory name
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_rsid_map

## Create linear model

In [ ]:
# REDS covariates: Additive, Sex, Age, and BMI
# genotype = "i_rs7904932:71150946:C:T"
genotype = "i_rs10762276:71052469:C:T"
control_group = 0  # Homozygous reference
alpha = 0.05
covariates = {"AS", "Sex", "Age", "BMI"}
recompute_unadjusted = True

if group_key is not None:
    covariates.add(group_key)
genotype, covariates

### Set contrasts

In [ ]:
all_groups = sorted(df_all_data.loc[:, (f"Genomics-{GENE_ID}", genotype)].dropna().unique())
# Create contrast group labels
contrasts_dict = {
    f"Group[{g2}] - Group[{g1}]": (g2, g1)
    for g1, g2 in list(combinations(all_groups, 2))
    if g1 == control_group
}
contrasts_dict

### Fit models for Metabolomics

In [ ]:
data_types = ["Metabolomics"]

# Set temp directory
temp_dirpath = INTERIM_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
temp_dirpath.mkdir(parents=True, exist_ok=True)
# Set output directory
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
output_dirpath.mkdir(parents=True, exist_ok=True)

#### Fit models
##### Fit model without covariate adjustments

In [ ]:
if recompute_unadjusted:
    covars = {}
    temp_dirpath_covars = temp_dirpath / "-".join(
        ["Covariates"] + (sorted(covars) if covars else ["None"])
    )
    temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
    logger.debug(
        f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
    )
    columns = [
        (data_type, variable)
        for data_type in data_types
        for variable in df_all_data.loc[:, data_type].columns
    ]
    columns += [(f"Genomics-{GENE_ID}", genotype)]
    columns += [("Metadata", covar) for covar in covars] if covars else []
    df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

    results_dict = fit_limma_model_for_genotype(
        data=df_data.copy(),
        genotype=genotype,
        control_group=control_group,
        covariates=covars,
        contrasts=list(contrasts_dict),
    )
    # Save results
    for contrast, df in results_dict.items():
        filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
        if contrast != "Overall":
            filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
        output_filename = make_filename(*filename_args)
        save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)

##### Fit model with covariate adjustments

In [ ]:
# With adjustments
covars = covariates
temp_dirpath_covars = temp_dirpath / "-".join(
    ["Covariates"] + (sorted(covars) if covars else ["None"])
)
temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
logger.debug(
    f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
)
columns = [
    (data_type, variable)
    for data_type in data_types
    for variable in df_all_data.loc[:, data_type].columns
]
columns += [(f"Genomics-{GENE_ID}", genotype)]
columns += [("Metadata", covar) for covar in covars] if covars else []
df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

results_dict = fit_limma_model_for_genotype(
    data=df_data.copy(),
    genotype=genotype,
    control_group=control_group,
    covariates=covars,
    contrasts=list(contrasts_dict),
)
# Save results
for contrast, df in results_dict.items():
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast != "Overall":
        filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
    output_filename = make_filename(*filename_args)
    save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)
    print(contrast)

#### Combine unadjusted and covariate adjusted data

In [ ]:
output_dirpath_covars = output_dirpath / "-".join(["Covariates"] + sorted(covariates))
output_dirpath_covars.mkdir(exist_ok=True, parents=True)

for contrast in [None] + list(contrasts_dict.values()):
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast is not None:
        filename_args += ["{1}_vs_{0}".format(*contrast)]
    # Set filename and shared dirpath
    input_filename = make_filename(*filename_args)
    # Load unadjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates", "None"])
    df_unadjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
    # Load covariate adjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates"] + sorted(covariates))
    df_adjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)

    # Create DataFrame for p-values
    compare_results_pvalues = pd.DataFrame.from_dict(
        {
            "pvalue no covariate adjustment": df_unadjusted.pvalue,
            "pvalue covariate adjustment": df_adjusted.pvalue,
            "FDR no covariate adjustment": df_unadjusted.adj_pvalue,
            "FDR covariate adjustment": df_adjusted.adj_pvalue,
        },
        orient="columns",
    )
    output_filename = make_filename(*filename_args, "pvalues")
    save_dataframe_to_file(
        compare_results_pvalues, output_dirpath_covars / output_filename, index=True
    )
compare_results_pvalues.sort_values("FDR covariate adjustment")

### Fit models for Proteomics

In [ ]:
data_types = ["Proteomics"]

# Set temp directory
temp_dirpath = INTERIM_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
temp_dirpath.mkdir(parents=True, exist_ok=True)
# Set output directory
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
output_dirpath.mkdir(parents=True, exist_ok=True)

#### Fit models
##### Fit model without covariate adjustments

In [ ]:
if recompute_unadjusted:
    covars = {}
    temp_dirpath_covars = temp_dirpath / "-".join(
        ["Covariates"] + (sorted(covars) if covars else ["None"])
    )
    temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
    logger.debug(
        f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
    )
    columns = [
        (data_type, variable)
        for data_type in data_types
        for variable in df_all_data.loc[:, data_type].columns
    ]
    columns += [(f"Genomics-{GENE_ID}", genotype)]
    columns += [("Metadata", covar) for covar in covars] if covars else []
    df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

    results_dict = fit_limma_model_for_genotype(
        data=df_data.copy(),
        genotype=genotype,
        control_group=control_group,
        covariates=covars,
        contrasts=list(contrasts_dict),
    )
    # Save results
    for contrast, df in results_dict.items():
        filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
        if contrast != "Overall":
            filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
        output_filename = make_filename(*filename_args)
        save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)

##### Fit model with covariate adjustments

In [ ]:
# With adjustments
covars = covariates
temp_dirpath_covars = temp_dirpath / "-".join(
    ["Covariates"] + (sorted(covars) if covars else ["None"])
)
temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
logger.debug(
    f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
)
columns = [
    (data_type, variable)
    for data_type in data_types
    for variable in df_all_data.loc[:, data_type].columns
]
columns += [(f"Genomics-{GENE_ID}", genotype)]
columns += [("Metadata", covar) for covar in covars] if covars else []
df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

results_dict = fit_limma_model_for_genotype(
    data=df_data.copy(),
    genotype=genotype,
    control_group=control_group,
    covariates=covars,
    contrasts=list(contrasts_dict),
)
# Save results
for contrast, df in results_dict.items():
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast != "Overall":
        filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
    output_filename = make_filename(*filename_args)
    save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)

#### Combine unadjusted and covariate adjusted data

In [ ]:
output_dirpath_covars = output_dirpath / "-".join(["Covariates"] + sorted(covariates))
output_dirpath_covars.mkdir(exist_ok=True, parents=True)

for contrast in [None] + list(contrasts_dict.values()):
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast is not None:
        filename_args += ["{1}_vs_{0}".format(*contrast)]
    # Set filename and shared dirpath
    input_filename = make_filename(*filename_args)
    # Load unadjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates", "None"])
    df_unadjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
    # Load covariate adjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates"] + sorted(covariates))
    df_adjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)

    # Create DataFrame for p-values
    compare_results_pvalues = pd.DataFrame.from_dict(
        {
            "pvalue no covariate adjustment": df_unadjusted.pvalue,
            "pvalue covariate adjustment": df_adjusted.pvalue,
            "FDR no covariate adjustment": df_unadjusted.adj_pvalue,
            "FDR covariate adjustment": df_adjusted.adj_pvalue,
        },
        orient="columns",
    )
    output_filename = make_filename(*filename_args, "pvalues")
    save_dataframe_to_file(
        compare_results_pvalues, output_dirpath_covars / output_filename, index=True
    )

### Fit models for Lipidomics

In [ ]:
data_types = ["Lipidomics"]

# Set temp directory
temp_dirpath = INTERIM_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
temp_dirpath.mkdir(parents=True, exist_ok=True)
# Set output directory
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "LinearModel" / "-".join([source_type] + data_types)
output_dirpath.mkdir(parents=True, exist_ok=True)

#### Fit models
##### Fit model without covariate adjustments

In [ ]:
if recompute_unadjusted:
    covars = {}
    temp_dirpath_covars = temp_dirpath / "-".join(
        ["Covariates"] + (sorted(covars) if covars else ["None"])
    )
    temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
    logger.debug(
        f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
    )
    columns = [
        (data_type, variable)
        for data_type in data_types
        for variable in df_all_data.loc[:, data_type].columns
    ]
    columns += [(f"Genomics-{GENE_ID}", genotype)]
    columns += [("Metadata", covar) for covar in covars] if covars else []
    df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

    results_dict = fit_limma_model_for_genotype(
        data=df_data.copy(),
        genotype=genotype,
        control_group=control_group,
        covariates=covars,
        contrasts=list(contrasts_dict),
    )
    # Save results
    for contrast, df in results_dict.items():
        filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
        if contrast != "Overall":
            filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
        output_filename = make_filename(*filename_args)
        save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)

##### Fit model with covariate adjustments

In [ ]:
# With adjustments
covars = covariates
temp_dirpath_covars = temp_dirpath / "-".join(
    ["Covariates"] + (sorted(covars) if covars else ["None"])
)
temp_dirpath_covars.mkdir(exist_ok=True, parents=True)
logger.debug(
    f"Formatting data with genotype '{genotype}'; covariates: '{', '.join(covars)}';  data type: '{', '.join(data_types)}'."
)
columns = [
    (data_type, variable)
    for data_type in data_types
    for variable in df_all_data.loc[:, data_type].columns
]
columns += [(f"Genomics-{GENE_ID}", genotype)]
columns += [("Metadata", covar) for covar in covars] if covars else []
df_data = df_all_data.loc[:, columns].droplevel(0, axis=1).copy()

results_dict = fit_limma_model_for_genotype(
    data=df_data.copy(),
    genotype=genotype,
    control_group=control_group,
    covariates=covars,
    contrasts=list(contrasts_dict),
)
# Save results
for contrast, df in results_dict.items():
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast != "Overall":
        filename_args += ["{1}_vs_{0}".format(*contrasts_dict[contrast])]
    output_filename = make_filename(*filename_args)
    save_dataframe_to_file(df, temp_dirpath_covars / output_filename, index=True)

#### Combine unadjusted and covariate adjusted data

In [ ]:
output_dirpath_covars = output_dirpath / "-".join(["Covariates"] + sorted(covariates))
output_dirpath_covars.mkdir(exist_ok=True, parents=True)

for contrast in [None] + list(contrasts_dict.values()):
    filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype.replace(":", "."))]
    if contrast is not None:
        filename_args += ["{1}_vs_{0}".format(*contrast)]
    # Set filename and shared dirpath
    input_filename = make_filename(*filename_args)
    # Load unadjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates", "None"])
    df_unadjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
    # Load covariate adjusted data
    input_dirpath = temp_dirpath / "-".join(["Covariates"] + sorted(covariates))
    df_adjusted = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)

    # Create DataFrame for p-values
    compare_results_pvalues = pd.DataFrame.from_dict(
        {
            "pvalue no covariate adjustment": df_unadjusted.pvalue,
            "pvalue covariate adjustment": df_adjusted.pvalue,
            "FDR no covariate adjustment": df_unadjusted.adj_pvalue,
            "FDR covariate adjustment": df_adjusted.adj_pvalue,
        },
        orient="columns",
    )
    output_filename = make_filename(*filename_args, "pvalues")
    save_dataframe_to_file(
        compare_results_pvalues, output_dirpath_covars / output_filename, index=True
    )